# Pretrained forecaster: Chronos-T5-small, zero-tuned and fine-tuned

Part of the reference implementation for *Reliability-Aware Model-Dependent Uncertainty Placement under Data Asymmetry for Probabilistic Load Forecasting*. Both uncertainty placements are produced: the modular placement (post-hoc contextual residual quantiles, route 1) and the integrated placement (in-model quantiles, route 2).

Run the cells in order. Outputs go to `outputs/`. A GPU is required for fine-tuning (about 34 min symmetric and 21 min asymmetric on one T4).

## 0. Environment

In [ ]:
# CLEAN INSTALL CELL FOR COLAB
!pip uninstall -y torchaudio torchvision torchtext
!pip cache purge

# install forecasting packages only
!pip install -q chronos-forecasting autogluon.timeseries
import torch
print("torch:", torch.__version__)

from chronos import BaseChronosPipeline
from autogluon.timeseries import TimeSeriesPredictor
print("Chronos import OK")
print("AutoGluon import OK")

## 1. Symmetric protocol ($X = X'$)
Training and testing at 5-minute resolution with all channels. Writes `route1` (point + modular quantiles) and `route2` (integrated quantiles) CSVs for the zero-tuned (LLM (ZT)) and the fine-tuned (LLM (FT)) Chronos-T5-small for Dec 24 and Dec 30, 2024.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# LLM symmetric evaluation at 5-minute resolution
# FIXED VERSION — key changes vs original:
#   FIX 1: Removed MAX_TRAIN_ROWS cap — full V1 data used for FT training
#   FIX 2: FT test uses direct 288-step forecast (not recursive 1-step rollout)
#   FIX 3: FT validation residuals computed from full-horizon day forecasts
#   FIX 4: time_limit raised to 7200 + fit_summary check
#   FIX 5: build_known_covariates uses positional assignment (kept from original)
# ============================


# !pip -q install -U autogluon.timeseries chronos-forecasting

import os
import gc
import time
import shutil
import random
import warnings
import logging
import contextlib
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# =========================
# 0) Silence noisy logging
# =========================
def silence_loggers():
    noisy_prefixes = [
        "autogluon", "gluonts", "lightning",
        "pytorch_lightning", "transformers", "chronos",
    ]
    root = logging.getLogger()
    root.handlers = []
    root.setLevel(logging.ERROR)
    for name in list(logging.root.manager.loggerDict.keys()):
        if any(name.startswith(p) for p in noisy_prefixes):
            lg = logging.getLogger(name)
            lg.handlers = []
            lg.propagate = False
            lg.setLevel(logging.ERROR)

silence_loggers()

@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, "w") as devnull:
        old_stdout = __import__("sys").stdout
        old_stderr = __import__("sys").stderr
        try:
            __import__("sys").stdout = devnull
            __import__("sys").stderr = devnull
            yield
        finally:
            __import__("sys").stdout = old_stdout
            __import__("sys").stderr = old_stderr

# =========================
# 1) Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# 2) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/llm_out_symmetric_5min_fixed1"
os.makedirs(OUT_DIR, exist_ok=True)

LOCAL_TRAIN_DIR = OUT_ROOT + "/_autogluon_chronos_trained_5min_fixed1_finetuned"

# =========================
# 3) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ      = "5min"

# =========================
# 4) Forecast settings
# =========================
STEPS_PER_HOUR = 12
HORIZON_HOURS  = 24
PRED_LEN       = HORIZON_HOURS * STEPS_PER_HOUR  # 288

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90

# ZS Chronos
MODEL_ZS          = "amazon/chronos-t5-small"
CONTEXT_LEN       = 24 * 12   # 288 = 24h
NUM_SAMPLES_ZS_R1 = 40
NUM_SAMPLES_ZS_R2 = 40

# FT Chronos
MODEL_FT_BASE = "amazon/chronos-t5-small"   # same base model as the zero-shot forecaster
FT_STEPS = 1000   # AutoGluon default is fine_tune=False, so set it explicitly
FT_LR = 1e-5
FT_CONTEXT_LEN = 24 * 12   # 288 = 24h

VAL_STRIDE = 1  # used only for ZS R1 residuals

# Output files
MERGED_R1_ZS = os.path.join(OUT_DIR, "merged_route1_point_uq_series_llm_zeroshot_symmetric_5min.csv")
MERGED_R2_ZS = os.path.join(OUT_DIR, "merged_route2_predictions_prob_llm_zeroshot_symmetric_5min.csv")
MERGED_R1_FT = os.path.join(OUT_DIR, "merged_route1_point_uq_series_llm_trained_symmetric_5min.csv")
MERGED_R2_FT = os.path.join(OUT_DIR, "merged_route2_predictions_prob_llm_trained_symmetric_5min.csv")



# =========================
# 5) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    out = pd.DataFrame({
        "timestamp":        merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)":          merged["q10"],
        "q50 (W)":          merged["q50"],
        "q90 (W)":          merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {"power_w": p_col, "temp_c": t_col, "voltage_v": v_col,
                "amperage_a": a_col, "generation_w": g_col}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found: {df.columns.tolist()}")

    print(f"[{os.path.basename(path)}] power='{p_col}', temp='{t_col}', "
          f"voltage='{v_col}', amperage='{a_col}', generation='{g_col}'")

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)
    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w", t_col: "temp_c", v_col: "voltage_v",
        a_col: "amperage_a", g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v",
        "amperage_a", "generation_w"]].copy()

    out["hour"]          = out["timestamp"].dt.hour
    out["dayofweek"]     = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"]     = out["minute_of_day"] // 5
    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(start=df["timestamp"].min(),
                              end=df["timestamp"].max(), freq="5min")
    df = (df.set_index("timestamp").reindex(full_idx)
            .rename_axis("timestamp").reset_index())

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"]          = df["timestamp"].dt.hour
    df["dayofweek"]     = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"]     = df["minute_of_day"] // 5
    return df

def add_engineered_features_5min(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)
    df["lag_5min"]  = df["power_w"].shift(1)
    df["lag_10min"] = df["power_w"].shift(2)
    df["lag_1h"]    = df["power_w"].shift(12)
    df["lag_24h"]   = df["power_w"].shift(288)
    df["ma_1h"]     = df["power_w"].shift(1).rolling(window=12,  min_periods=1).mean()
    df["ma_3h"]     = df["power_w"].shift(1).rolling(window=36,  min_periods=1).mean()
    df = df.bfill().ffill()
    return df

def apply_route1_buckets_5min(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["slot_5min"] = tmp["timestamp"].dt.hour * 12 + (tmp["timestamp"].dt.minute // 5)
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek
    tmp = tmp.merge(res_q, on=["slot_5min", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()
    return enforce_non_crossing(p10, p50, p90)

def save_route1_day(test_df, y_true, point_pred, p10, p50b, p90b, path):
    out = pd.DataFrame({
        "timestamp":  test_df["timestamp"].values,
        "actual":     y_true,
        "point_pred": point_pred,
        "q10": p10, "q50": p50b, "q90": p90b,
    })
    out.to_csv(path, index=False)

def save_route2_day(test_df, y_true, q10, q50, q90, path):
    out = pd.DataFrame({
        "timestamp": test_df["timestamp"].values,
        "actual": y_true,
        "q10": q10, "q50": q50, "q90": q90,
    })
    out.to_csv(path, index=False)

# =========================
# 6) Load data
# =========================
print("Loading V1/V2 at 5-minute resolution...")

v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)
full_5m = add_engineered_features_5min(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

print("✅ V1 5-min:", len(v1_5m))
print("✅ V2 5-min:", len(v2_5m))

# =========================
# 7) Symmetric split
# =========================
test_mask   = v2_5m["timestamp"].dt.normalize().isin(TEST_DAYS)
train_df    = v1_5m.copy().reset_index(drop=True)
val_df      = v2_5m[~test_mask].copy().reset_index(drop=True)
test_df_all = v2_5m[test_mask].copy().reset_index(drop=True)

print("✅ Train rows:", len(train_df))
print("✅ Val rows  :", len(val_df))
print("✅ Test rows :", len(test_df_all))

for d in TEST_DAYS:
    cur = test_df_all[test_df_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != PRED_LEN:
        raise ValueError(f"Test day {d.date()} has {len(cur)} rows, expected {PRED_LEN}.")

# =========================
# 8) Shared FT helpers
# =========================
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

TARGET_COL       = "power_w"
KNOWN_COVARIATES = [
    "temp_c", "voltage_v", "amperage_a", "generation_w",
    "hour", "dayofweek", "minute_of_day", "slot_5min",
]

def build_hist_ts_from_df(df):
    tmp = df.copy().rename(columns={TARGET_COL: "target"})
    tmp["item_id"] = "series_1"
    return TimeSeriesDataFrame.from_data_frame(
        tmp[["item_id", "timestamp", "target"] + KNOWN_COVARIATES],
        id_column="item_id",
        timestamp_column="timestamp"
    )

def build_known_covariates_from_future_df(predictor, hist_ts,
                                           source_future_df,
                                           known_covariate_cols):
    """
    Build the known-covariates TimeSeriesDataFrame for prediction.
    Uses positional assignment — no timestamp merge — to avoid NaN covariate bug.
    """
    freq = pd.tseries.frequencies.to_offset(SRC_FREQ)

    last_ts = hist_ts.index.get_level_values("timestamp").max()

    future_timestamps = pd.date_range(
        start=last_ts + freq,
        periods=PRED_LEN,
        freq=freq
    )

    src = source_future_df.copy().reset_index(drop=True)
    if len(src) < PRED_LEN:
        pad = src.iloc[[-1]].copy()
        while len(src) < PRED_LEN:
            src = pd.concat([src, pad], ignore_index=True)
    src = src.iloc[:PRED_LEN].reset_index(drop=True)

    future_cov = pd.DataFrame({"timestamp": future_timestamps})
    future_cov["item_id"] = "series_1"

    sensor_cols = ["temp_c", "voltage_v", "amperage_a", "generation_w"]
    for col in sensor_cols:
        if col in known_covariate_cols and col in src.columns:
            future_cov[col] = src[col].values
        else:
            future_cov[col] = 0.0

    future_cov["hour"]          = future_cov["timestamp"].dt.hour
    future_cov["dayofweek"]     = future_cov["timestamp"].dt.dayofweek
    future_cov["minute_of_day"] = (future_cov["timestamp"].dt.hour * 60
                                   + future_cov["timestamp"].dt.minute)
    future_cov["slot_5min"]     = future_cov["minute_of_day"] // 5

    missing = future_cov[known_covariate_cols].isna().sum()
    if missing.sum() > 0:
        raise ValueError(f"Still missing covariates after fix:\n{missing[missing > 0]}")

    return TimeSeriesDataFrame.from_data_frame(
        future_cov[["item_id", "timestamp"] + known_covariate_cols],
        id_column="item_id",
        timestamp_column="timestamp"
    )

# =========================
# 9) PART A: ZS Chronos-T5
# (unchanged — ZS was working correctly)
# =========================
if os.path.exists(MERGED_R1_ZS) and os.path.exists(MERGED_R2_ZS):
    print("\n✅ ZS merged outputs already exist. Skipping ZS.")
else:
    print("\n" + "=" * 80)
    print("PART A) ZS symmetric: Chronos-T5")

    from chronos import BaseChronosPipeline

    device_map = "cuda" if torch.cuda.is_available() else "cpu"
    dtype      = torch.bfloat16 if device_map == "cuda" else torch.float32

    pipe_zs = BaseChronosPipeline.from_pretrained(
        MODEL_ZS, device_map=device_map, torch_dtype=dtype)
    print(f"✅ Chronos ZS loaded: {MODEL_ZS} | device={device_map}")

    def zs_forward_one_step_validation_predictions(pipe, train_df, val_df,
                                                    context_len,
                                                    num_samples=40, stride=1):
        hist_vals = train_df["power_w"].to_numpy(dtype=np.float32).tolist()
        val_vals  = val_df["power_w"].to_numpy(dtype=np.float32)
        val_ts    = pd.to_datetime(val_df["timestamp"]).to_numpy()

        if len(hist_vals) < context_len:
            raise ValueError(f"Need at least {context_len} history points.")

        preds, actuals, ts_list = [], [], []

        for j in range(len(val_df)):
            if j % 1000 == 0:
                print(f"  ZS validation rolling progress: {j}/{len(val_df)}")

            ctx   = np.asarray(hist_vals[-context_len:], dtype=np.float32)
            ctx_t = torch.tensor(ctx, dtype=torch.float32)

            with torch.no_grad():
                fc = pipe.predict(ctx_t, prediction_length=1,
                                  num_samples=num_samples)

            samp  = fc[0].cpu().numpy()
            yhat  = float(np.quantile(samp[:, 0], Q_MID))
            ytrue = float(val_vals[j])

            if (j % stride) == 0:
                preds.append(yhat)
                actuals.append(ytrue)
                ts_list.append(pd.Timestamp(val_ts[j]))

            hist_vals.append(ytrue)

        return pd.DataFrame({"timestamp": ts_list,
                              "actual": actuals, "pred": preds})

    def zs_recursive_test_forecast(pipe, hist_values, horizon,
                                    context_len, num_samples=40):
        hist_vals = list(hist_values)
        q10_list, q50_list, q90_list = [], [], []

        for h in range(horizon):
            ctx   = np.asarray(hist_vals[-context_len:], dtype=np.float32)
            ctx_t = torch.tensor(ctx, dtype=torch.float32)

            with torch.no_grad():
                fc = pipe.predict(ctx_t, prediction_length=1,
                                  num_samples=num_samples)

            samp = fc[0].cpu().numpy()
            q10  = float(np.quantile(samp[:, 0], Q_LO))
            q50  = float(np.quantile(samp[:, 0], Q_MID))
            q90  = float(np.quantile(samp[:, 0], Q_HI))

            aa, bb, cc = enforce_non_crossing(
                np.array([q10]), np.array([q50]), np.array([q90]))
            q10, q50, q90 = float(aa[0]), float(bb[0]), float(cc[0])

            q10_list.append(q10)
            q50_list.append(q50)
            q90_list.append(q90)
            hist_vals.append(q50)

        return (np.asarray(q10_list, dtype=float),
                np.asarray(q50_list, dtype=float),
                np.asarray(q90_list, dtype=float))

    print("Building standardized ZS Route1 residual buckets from validation...")
    df_resid_zs = zs_forward_one_step_validation_predictions(
        pipe=pipe_zs, train_df=train_df, val_df=val_df,
        context_len=CONTEXT_LEN,
        num_samples=NUM_SAMPLES_ZS_R1, stride=VAL_STRIDE)

    df_resid_zs["residual"] = df_resid_zs["actual"] - df_resid_zs["pred"]
    clip_val = np.percentile(np.abs(df_resid_zs["residual"].to_numpy()), 99)
    df_resid_zs["residual"] = np.clip(df_resid_zs["residual"], -clip_val, clip_val)
    df_resid_zs["slot_5min"] = (df_resid_zs["timestamp"].dt.hour * 12
                                + (df_resid_zs["timestamp"].dt.minute // 5))
    df_resid_zs["dayofweek"] = df_resid_zs["timestamp"].dt.dayofweek

    res_q_zs = (
        df_resid_zs.groupby(["slot_5min", "dayofweek"])["residual"]
                   .quantile([Q_LO, Q_MID, Q_HI]).unstack()
                   .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
                   .reset_index()
    )

    df_resid_zs.to_csv(
        os.path.join(OUT_DIR, "chronos_zeroshot_route1_residuals_from_v2val_5min.csv"),
        index=False)
    res_q_zs.to_csv(
        os.path.join(OUT_DIR, "chronos_zeroshot_route1_residual_quantiles_from_v2val_5min.csv"),
        index=False)

    r1_day_files_zs, r2_day_files_zs = [], []

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print(f"ZS test day {k}: {day_start.date()}")
        day_end  = day_start + pd.Timedelta(days=1)
        test_df  = test_df_all[
            (test_df_all["timestamp"] >= day_start) &
            (test_df_all["timestamp"] <  day_end)
        ].copy().reset_index(drop=True)

        y_true     = test_df["power_w"].to_numpy(dtype=float)
        cutoff     = day_start - pd.Timedelta(minutes=5)
        hist_slice = full_5m[full_5m["timestamp"] <= cutoff].copy().reset_index(drop=True)

        if len(hist_slice) < CONTEXT_LEN:
            raise ValueError(f"Not enough context before {day_start}.")

        hist_values = hist_slice["power_w"].to_numpy(dtype=np.float32).tolist()

        q10, q50, q90 = zs_recursive_test_forecast(
            pipe=pipe_zs, hist_values=hist_values, horizon=PRED_LEN,
            context_len=CONTEXT_LEN, num_samples=NUM_SAMPLES_ZS_R2)

        p_r2 = os.path.join(OUT_DIR, f"zeroshot_route2_symmetric_5min_day{k}.csv")
        save_route2_day(test_df, y_true, q10, q50, q90, p_r2)
        r2_day_files_zs.append(p_r2)

        point_pred = q50.copy()
        p10, p50b, p90b = apply_route1_buckets_5min(
            test_df["timestamp"].values, point_pred, res_q_zs)
        p_r1 = os.path.join(OUT_DIR, f"zeroshot_route1_symmetric_5min_day{k}.csv")
        save_route1_day(test_df, y_true, point_pred, p10, p50b, p90b, p_r1)
        r1_day_files_zs.append(p_r1)

    merge_daily_route1(r1_day_files_zs, MERGED_R1_ZS)
    merge_daily_route2(r2_day_files_zs, MERGED_R2_ZS)
    print("✅ ZS done")

    del pipe_zs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =========================
# 10) PART B: FT Chronos-T5-small fine-tuned via AutoGluon
# FIXED:
#   - No MAX_TRAIN_ROWS cap: full V1 used
#   - time_limit raised to 7200 with fit_summary check
#   - Validation residuals from full-horizon day-level forecasts
#   - Test forecast uses single direct 288-step call per day
# =========================
if os.path.exists(MERGED_R1_FT) and os.path.exists(MERGED_R2_FT):
    print("\n✅ FT merged outputs already exist. Skipping FT.")
else:
    print("\n" + "=" * 80)
    print("PART B) FT symmetric: Chronos-T5-small fine-tuned via AutoGluon [FIXED]")

    # ----------------------------------------------------------
    # Try to reload saved model first; retrain only if missing
    # ----------------------------------------------------------
    if os.path.exists(LOCAL_TRAIN_DIR) and os.path.isdir(LOCAL_TRAIN_DIR):
        try:
            predictor  = TimeSeriesPredictor.load(LOCAL_TRAIN_DIR)
            best_model = predictor.model_best
            print(f"✅ Reloaded saved FT model from {LOCAL_TRAIN_DIR} | best={best_model}")
        except Exception as e:
            print(f"⚠️  Could not reload saved model ({repr(e)}). Retraining...")
            predictor = None
    else:
        predictor = None

    if predictor is None:
        if os.path.exists(LOCAL_TRAIN_DIR):
            shutil.rmtree(LOCAL_TRAIN_DIR)
        os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

        # -----------------------------------------------------------
        # FIX 1: Use full training data — NO MAX_TRAIN_ROWS cap.
        # The original code truncated to 10k rows (~35 days), throwing
        # away ~90% of V1. Chronos needs the full year to learn
        # meaningful seasonal and weekly patterns.
        # -----------------------------------------------------------
        train_df_ft = train_df.copy()
        print(f"  Using full training set: {len(train_df_ft)} rows "
              f"({len(train_df_ft)/STEPS_PER_HOUR/24:.1f} days)")

        train_ts = build_hist_ts_from_df(train_df_ft)

        print("Fitting AutoGluon TimeSeriesPredictor...")
        predictor = TimeSeriesPredictor(
            prediction_length=PRED_LEN,
            freq=SRC_FREQ,
            path=LOCAL_TRAIN_DIR,
            target="target",
            eval_metric="MASE",
            known_covariates_names=KNOWN_COVARIATES,
            quantile_levels=[Q_LO, Q_MID, Q_HI],
            verbosity=2,
        )

        t_fit = time.time()
        try:
            predictor.fit(
                train_data=train_ts,
                hyperparameters={"Chronos": {"model_path": MODEL_FT_BASE, "fine_tune": True, "fine_tune_steps": FT_STEPS, "fine_tune_lr": FT_LR}},
                enable_ensemble=False,
                # FIX 4: Raised from 3600 → 7200 to give Chronos
                # enough time to fine-tune on the full dataset.
                time_limit=7200,
            )
            print(f"✅ FT fit completed in {time.time()-t_fit:.1f}s")
        except Exception as e:
            print(f"❌ FT fit failed: {repr(e)}")
            raise

        trained_models = predictor.model_names()
        if not trained_models:
            raise RuntimeError("No models were trained successfully.")

        best_model = predictor.model_best
        elapsed    = time.time() - t_fit
        print(f"✅ FT trained in {elapsed:.1f}s | best={best_model}")

        # FIX 4 cont.: Warn if time limit was the binding constraint
        summary = predictor.fit_summary()
        print("\n--- fit_summary ---")
        print(summary)
        if elapsed >= 7000:
            print("⚠️  WARNING: training hit the time limit. "
                  "Consider increasing time_limit further.")

    # ----------------------------------------------------------
    # Residual buckets — FIX 3:
    # Computed from full-horizon (288-step) day-level forecasts
    # over the validation set, matching what we do at test time.
    # Original code used recursive 1-step-ahead which mismatch the
    # FT model's actual multi-step output distribution.
    # ----------------------------------------------------------
    RES_Q_FT_PATH = os.path.join(OUT_DIR, "chronos_trained_route1_residual_quantiles_from_v2val_5min.csv")
    RESID_FT_PATH = os.path.join(OUT_DIR, "chronos_trained_route1_residuals_from_v2val_5min.csv")

    def ft_day_level_validation_predictions(predictor, best_model,
                                             train_df, val_df, context_len):
        """
        FIX 3: For each calendar day in val_df, make a single 288-step
        direct forecast (no recursive rollout).  Residuals are therefore
        drawn from the same distribution as the test forecasts.

        history is accumulated day-by-day using true observed values,
        not the model's own predictions, so there is no error compounding.
        """
        val_df = val_df.copy().sort_values("timestamp").reset_index(drop=True)
        val_days = sorted(val_df["timestamp"].dt.normalize().unique())

        hist = train_df.copy().sort_values("timestamp").reset_index(drop=True)

        all_ts, all_actual, all_pred = [], [], []

        for d_idx, day_start in enumerate(val_days):
            day_end = day_start + pd.Timedelta(days=1)
            day_df  = val_df[
                (val_df["timestamp"] >= day_start) &
                (val_df["timestamp"] <  day_end)
            ].copy().reset_index(drop=True)

            if len(day_df) < PRED_LEN:
                # Partial day — skip to avoid misaligned residuals
                hist = pd.concat([hist, day_df], ignore_index=True)
                hist = hist.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
                continue

            if d_idx % 5 == 0:
                print(f"  FT val day forecast {d_idx+1}/{len(val_days)}: {day_start.date()}")

            hist_for_model = hist.tail(context_len).copy().reset_index(drop=True)
            hist_ts        = build_hist_ts_from_df(hist_for_model)

            source_future_df           = day_df.copy().reset_index(drop=True)
            source_future_df["item_id"] = "series_1"

            try:
                known_cov_ts = build_known_covariates_from_future_df(
                    predictor=predictor, hist_ts=hist_ts,
                    source_future_df=source_future_df,
                    known_covariate_cols=KNOWN_COVARIATES)
            except ValueError as e:
                print(f"  Skipping day {day_start.date()}: {e}")
                hist = pd.concat([hist, day_df], ignore_index=True)
                hist = hist.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
                continue

            with suppress_output():
                fc = predictor.predict(
                    hist_ts, known_covariates=known_cov_ts, model=best_model)

            fc_pd = fc.reset_index().sort_values("timestamp").reset_index(drop=True)

            # Align predictions to actual timestamps
            for i in range(min(PRED_LEN, len(day_df), len(fc_pd))):
                all_ts.append(pd.Timestamp(day_df.loc[i, "timestamp"]))
                all_actual.append(float(day_df.loc[i, "power_w"]))
                all_pred.append(float(fc_pd.loc[i, str(Q_MID)]))

            # Accumulate TRUE observed values into history (no error compounding)
            hist = pd.concat([hist, day_df], ignore_index=True)
            hist = hist.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

        return pd.DataFrame({"timestamp": all_ts,
                              "actual":    all_actual,
                              "pred":      all_pred})

    if os.path.exists(RES_Q_FT_PATH) and os.path.exists(RESID_FT_PATH):
        print("\n✅ FT residual buckets already exist. Reloading...")
        res_q_ft    = pd.read_csv(RES_Q_FT_PATH)
        df_resid_ft = pd.read_csv(RESID_FT_PATH)
        print(f"   Loaded res_q_ft: {len(res_q_ft)} rows")
    else:
        print("\nBuilding FT Route1 residual buckets from validation (day-level forecasts)...")
        df_resid_ft = ft_day_level_validation_predictions(
            predictor=predictor, best_model=best_model,
            train_df=train_df, val_df=val_df,
            context_len=FT_CONTEXT_LEN)

        df_resid_ft["timestamp"] = pd.to_datetime(df_resid_ft["timestamp"])
        df_resid_ft["residual"]  = df_resid_ft["actual"] - df_resid_ft["pred"]
        clip_val = np.percentile(np.abs(df_resid_ft["residual"].to_numpy()), 99)
        df_resid_ft["residual"] = np.clip(df_resid_ft["residual"], -clip_val, clip_val)
        df_resid_ft["slot_5min"] = (df_resid_ft["timestamp"].dt.hour * 12
                                    + df_resid_ft["timestamp"].dt.minute // 5)
        df_resid_ft["dayofweek"] = df_resid_ft["timestamp"].dt.dayofweek

        res_q_ft = (
            df_resid_ft.groupby(["slot_5min", "dayofweek"])["residual"]
                       .quantile([Q_LO, Q_MID, Q_HI]).unstack()
                       .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
                       .reset_index()
        )

        df_resid_ft.to_csv(RESID_FT_PATH, index=False)
        res_q_ft.to_csv(RES_Q_FT_PATH, index=False)
        print("✅ Saved FT residual buckets (day-level)")

    # ----------------------------------------------------------
    # Direct test forecast — FIX 2:
    # One single 288-step predict() call per test day.
    # This is how Chronos is designed to be used.
    # The original recursive 1-step rollout forced the model to
    # repeatedly produce only the first step of a 288-step forecast,
    # which is both inefficient and misaligned with training.
    # ----------------------------------------------------------
    def ft_direct_test_forecast(predictor, best_model,
                                 initial_hist_df, test_df, context_len):
        """
        FIX 2: Single direct 288-step forecast for the full test day.
        Returns q10, q50, q90 arrays of length PRED_LEN.
        """
        hist_for_model = initial_hist_df.tail(context_len).copy().reset_index(drop=True)
        hist_ts        = build_hist_ts_from_df(hist_for_model)

        source_future_df           = test_df.copy().reset_index(drop=True)
        source_future_df["item_id"] = "series_1"

        known_cov_ts = build_known_covariates_from_future_df(
            predictor=predictor, hist_ts=hist_ts,
            source_future_df=source_future_df,
            known_covariate_cols=KNOWN_COVARIATES)

        with suppress_output():
            fc = predictor.predict(
                hist_ts, known_covariates=known_cov_ts, model=best_model)

        fc_pd = fc.reset_index().sort_values("timestamp").reset_index(drop=True)

        if len(fc_pd) != PRED_LEN:
            raise ValueError(
                f"Expected {PRED_LEN} forecast rows, got {len(fc_pd)}. "
                "Check that prediction_length matches PRED_LEN.")

        q10 = fc_pd[str(Q_LO)].to_numpy(dtype=float)
        q50 = fc_pd[str(Q_MID)].to_numpy(dtype=float)
        q90 = fc_pd[str(Q_HI)].to_numpy(dtype=float)

        return enforce_non_crossing(q10, q50, q90)

    r1_day_files_ft, r2_day_files_ft = [], []

    # hist_master includes everything up to (but not including) test days
    hist_master = pd.concat([train_df, val_df], ignore_index=True)
    hist_master = (hist_master.sort_values("timestamp")
                              .drop_duplicates("timestamp")
                              .reset_index(drop=True))

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print(f"\nFT test day {k}: {day_start.date()}")
        day_end  = day_start + pd.Timedelta(days=1)
        test_df  = test_df_all[
            (test_df_all["timestamp"] >= day_start) &
            (test_df_all["timestamp"] <  day_end)
        ].copy().reset_index(drop=True)

        y_true     = test_df["power_w"].to_numpy(dtype=float)
        cutoff     = day_start - pd.Timedelta(minutes=5)
        hist_slice = hist_master[
            hist_master["timestamp"] <= cutoff
        ].copy().reset_index(drop=True)

        print(f"  History rows available: {len(hist_slice)} "
              f"({len(hist_slice)/STEPS_PER_HOUR/24:.1f} days)")

        if len(hist_slice) < FT_CONTEXT_LEN:
            raise ValueError(f"Not enough context before {day_start}. "
                             f"Need {FT_CONTEXT_LEN}, have {len(hist_slice)}.")

        # FIX 2: single direct call — no recursive rollout
        q10, q50, q90 = ft_direct_test_forecast(
            predictor=predictor, best_model=best_model,
            initial_hist_df=hist_slice, test_df=test_df,
            context_len=FT_CONTEXT_LEN)

        # Route 2 — direct probabilistic output
        p_r2 = os.path.join(OUT_DIR, f"trained_route2_symmetric_5min_day{k}.csv")
        save_route2_day(test_df, y_true, q10, q50, q90, p_r2)
        r2_day_files_ft.append(p_r2)

        # Route 1 — point + post-hoc UQ from residual buckets
        point_pred = q50.copy()
        p10, p50b, p90b = apply_route1_buckets_5min(
            test_df["timestamp"].values, point_pred, res_q_ft)
        p_r1 = os.path.join(OUT_DIR, f"trained_route1_symmetric_5min_day{k}.csv")
        save_route1_day(test_df, y_true, point_pred, p10, p50b, p90b, p_r1)
        r1_day_files_ft.append(p_r1)

        print(f"✅ Day {k} saved")

    merge_daily_route1(r1_day_files_ft, MERGED_R1_FT)
    merge_daily_route2(r2_day_files_ft, MERGED_R2_FT)
    print("✅ FT done")

# =========================
# Done
# =========================
print("\nAll done.")
print("ZS Route1:", MERGED_R1_ZS)
print("ZS Route2:", MERGED_R2_ZS)
print("FT Route1:", MERGED_R1_FT)
print("FT Route2:", MERGED_R2_FT)

## 2. Asymmetric protocol ($X \neq X'$)
Training on the full multivariate history; deployment on hourly data with temperature and calendar features only. Voltage, current and PV generation are filled with hour-of-day means from $D_1$.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# LLM ASYMMETRIC evaluation
#
# Fixed + deployment-mismatch version
#
# Models:
#   - ZS: Chronos-T5
#   - FT: Chronos-T5-small fine-tuned via AutoGluon
#
# Asymmetric setup:
#   - Source data loaded from 5-minute V1 / V2
#   - Train history comes from V1
#   - Validation: V2 excluding Dec 24 and Dec 30
#   - Test: Dec 24 and Dec 30 from V2
#
# Deployment mismatch:
#   - evaluation is HOURLY
#   - only temperature + time covariates are available at deployment
#
# Route1:
#   - point forecast + post-hoc residual buckets from degraded hourly V2 validation
#
# Route2:
#   - probabilistic forecast directly from model quantiles / samples
#
# Outputs:
#   merged_route1_point_uq_series_llm_zeroshot_asymmetric_hourly.csv
#   merged_route2_predictions_prob_llm_zeroshot_asymmetric_hourly.csv
#   merged_route1_point_uq_series_llm_trained_asymmetric_hourly.csv
#   merged_route2_predictions_prob_llm_trained_asymmetric_hourly.csv
# ============================


import os
import gc
import sys
import time
import shutil
import random
import warnings
import logging
import contextlib
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# =========================
# 0) Silence noisy logging
# =========================
def silence_loggers():
    noisy_prefixes = [
        "autogluon",
        "gluonts",
        "lightning",
        "pytorch_lightning",
        "transformers",
        "chronos",
    ]
    root = logging.getLogger()
    root.handlers = []
    root.setLevel(logging.ERROR)

    for name in list(logging.root.manager.loggerDict.keys()):
        if any(name.startswith(p) for p in noisy_prefixes):
            lg = logging.getLogger(name)
            lg.handlers = []
            lg.propagate = False
            lg.setLevel(logging.ERROR)

silence_loggers()

@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout, sys.stderr = devnull, devnull
            yield
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr

# =========================
# 1) Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# 2) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/llm_out_asymmetric_hourly"
os.makedirs(OUT_DIR, exist_ok=True)

LOCAL_TRAIN_DIR = OUT_ROOT + "/_autogluon_chronos_trained_asymmetric_hourly_local_finetuned"

# =========================
# 3) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# =========================
# 4) Forecast settings
# =========================
PRED_LEN = 24  # hourly 24h horizon

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90

# ZS Chronos
MODEL_ZS = "amazon/chronos-t5-small"
CONTEXT_DAYS_HOURLY = 1
CONTEXT_LEN_HOURLY = CONTEXT_DAYS_HOURLY * 24
NUM_SAMPLES_ZS = 40

# FT Chronos
MODEL_FT_BASE = "amazon/chronos-t5-small"   # same base model as the zero-shot forecaster
FT_STEPS = 1000   # AutoGluon default is fine_tune=False, so set it explicitly
FT_LR = 1e-5

# Speed knobs
FT_CALIB_STRIDE = 1
FT_CALIB_MAX_WINDOWS = 1200
ZS_CALIB_STRIDE = 1

# =========================
# 5) Output files
# =========================
MERGED_R1_ZS = os.path.join(OUT_DIR, "merged_route1_point_uq_series_llm_zeroshot_asymmetric_hourly.csv")
MERGED_R2_ZS = os.path.join(OUT_DIR, "merged_route2_predictions_prob_llm_zeroshot_asymmetric_hourly.csv")
MERGED_R1_FT = os.path.join(OUT_DIR, "merged_route1_point_uq_series_llm_trained_asymmetric_hourly.csv")
MERGED_R2_FT = os.path.join(OUT_DIR, "merged_route2_predictions_prob_llm_trained_asymmetric_hourly.csv")



# =========================
# 6) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def maybe_limit_indices(indices, max_windows):
    if len(indices) <= max_windows:
        return indices
    idx = np.linspace(0, len(indices) - 1, max_windows).round().astype(int)
    idx = np.unique(idx)
    return [indices[i] for i in idx]

def save_route1_day(test_df, y_true, point_pred, p10, p50b, p90b, path):
    out = pd.DataFrame({
        "timestamp": test_df["timestamp"].values,
        "actual": y_true,
        "point_pred": point_pred,
        "q10": p10,
        "q50": p50b,
        "q90": p90b
    })
    out.to_csv(path, index=False)

def save_route2_day(test_df, y_true, q10, q50, q90, path):
    out = pd.DataFrame({
        "timestamp": test_df["timestamp"].values,
        "actual": y_true,
        "q10": q10,
        "q50": q50,
        "q90": q90
    })
    out.to_csv(path, index=False)

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found columns: {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out["hour"] = out["timestamp"].dt.hour
    out["dayofweek"] = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"] = out["minute_of_day"] // 5

    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    return df

def aggregate_to_hourly(df_5m):
    df = df_5m.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ts_hour"] = df["timestamp"].dt.floor("H")

    ag = (
        df.groupby("ts_hour", as_index=False)
          .agg({
              "power_w": "mean",
              "temp_c": "mean",
              "voltage_v": "mean",
              "amperage_a": "mean",
              "generation_w": "mean",
          })
          .rename(columns={"ts_hour": "timestamp"})
    )

    ag["hour"] = ag["timestamp"].dt.hour
    ag["dayofweek"] = ag["timestamp"].dt.dayofweek
    return ag

def build_route1_buckets_hourly(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["hour"] = tmp["timestamp"].dt.hour
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["hour", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

# =========================
# 7) Load data
# =========================
print("Loading V1/V2 at 5-minute resolution...")

v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

# convert to hourly deployment regime
v1_h = aggregate_to_hourly(v1_5m)
v2_h = aggregate_to_hourly(v2_5m)

print("✅ V1 hourly:", len(v1_h))
print("✅ V2 hourly:", len(v2_h))

# =========================
# 8) Asymmetric split
# =========================
test_mask = v2_h["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df = v1_h.copy().reset_index(drop=True)
val_df = v2_h[~test_mask].copy().reset_index(drop=True)
test_df_all = v2_h[test_mask].copy().reset_index(drop=True)

print("✅ Train rows:", len(train_df))
print("✅ Val rows  :", len(val_df))
print("✅ Test rows :", len(test_df_all))

for d in TEST_DAYS:
    cur = test_df_all[test_df_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != PRED_LEN:
        raise ValueError(f"Test day {d.date()} has {len(cur)} rows, expected {PRED_LEN}.")

# =========================
# 9) PART A: ZS Chronos-T5
# =========================
if os.path.exists(MERGED_R1_ZS) and os.path.exists(MERGED_R2_ZS):
    print("\n✅ ZS merged outputs already exist. Skipping ZS.")
else:
    print("\n" + "=" * 80)
    print("PART A) ZS asymmetric hourly: Chronos-T5")

    from chronos import BaseChronosPipeline

    device_map = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device_map == "cuda" else torch.float32

    pipe_zs = BaseChronosPipeline.from_pretrained(
        MODEL_ZS,
        device_map=device_map,
        torch_dtype=dtype
    )
    print(f"✅ Chronos ZS loaded: {MODEL_ZS} | device={device_map}")

    # ---- ZS Route1 residual buckets from degraded hourly V2 validation only
    rows_zs = []
    t0 = time.time()

    val_vals = val_df["power_w"].to_numpy(dtype=np.float32)
    val_ts = val_df["timestamp"].to_numpy()

    for j in range(CONTEXT_LEN_HOURLY, len(val_df), ZS_CALIB_STRIDE):
        ctx = val_vals[j - CONTEXT_LEN_HOURLY:j]
        y_t = float(val_vals[j])
        ts = pd.to_datetime(val_ts[j])

        ctx_t = torch.tensor(ctx, dtype=torch.float32)
        with torch.no_grad():
            fc = pipe_zs.predict(ctx_t, prediction_length=1, num_samples=NUM_SAMPLES_ZS)

        samp = fc[0].cpu().numpy()
        pred_q50 = float(np.quantile(samp[:, 0], Q_MID))

        rows_zs.append({
            "hour": int(ts.hour),
            "dayofweek": int(ts.dayofweek),
            "residual": float(y_t - pred_q50)
        })

    df_resid_zs = pd.DataFrame(rows_zs)
    print(f"✅ ZS residual points: {len(df_resid_zs)} in {time.time()-t0:.1f}s")

    res_q_zs = (
        df_resid_zs.groupby(["hour", "dayofweek"])["residual"]
                   .quantile([Q_LO, Q_MID, Q_HI])
                   .unstack()
                   .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
                   .reset_index()
    )
    res_q_zs.to_csv(os.path.join(OUT_DIR, "chronos_zeroshot_route1_residual_quantiles_from_v2val_asymmetric_hourly.csv"), index=False)

    r1_day_files_zs = []
    r2_day_files_zs = []

    hist_master_h = pd.concat([train_df, val_df, test_df_all], ignore_index=True)
    hist_master_h = hist_master_h.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print(f"ZS test day {k}: {day_start.date()}")
        day_end = day_start + pd.Timedelta(days=1)

        test_df = test_df_all[
            (test_df_all["timestamp"] >= day_start) &
            (test_df_all["timestamp"] < day_end)
        ].copy().reset_index(drop=True)

        y_true = test_df["power_w"].to_numpy(dtype=float)

        cutoff = day_start - pd.Timedelta(hours=1)
        hist_slice = hist_master_h[hist_master_h["timestamp"] <= cutoff].copy().reset_index(drop=True)

        if len(hist_slice) < CONTEXT_LEN_HOURLY:
            raise ValueError(f"Not enough hourly context before {day_start}. Have {len(hist_slice)} < {CONTEXT_LEN_HOURLY}")

        ctx = hist_slice["power_w"].to_numpy(dtype=np.float32)[-CONTEXT_LEN_HOURLY:]
        ctx_t = torch.tensor(ctx, dtype=torch.float32)

        with torch.no_grad():
            fc = pipe_zs.predict(ctx_t, prediction_length=PRED_LEN, num_samples=NUM_SAMPLES_ZS)

        samples = fc[0].cpu().numpy()
        q10 = np.quantile(samples, Q_LO, axis=0)
        q50 = np.quantile(samples, Q_MID, axis=0)
        q90 = np.quantile(samples, Q_HI, axis=0)
        q10, q50, q90 = enforce_non_crossing(q10, q50, q90)

        p_r2 = os.path.join(OUT_DIR, f"zeroshot_route2_asymmetric_hourly_day{k}.csv")
        save_route2_day(test_df, y_true, q10, q50, q90, p_r2)
        r2_day_files_zs.append(p_r2)

        point_pred = q50.copy()
        p10, p50b, p90b = build_route1_buckets_hourly(test_df["timestamp"].values, point_pred, res_q_zs)

        p_r1 = os.path.join(OUT_DIR, f"zeroshot_route1_asymmetric_hourly_day{k}.csv")
        save_route1_day(test_df, y_true, point_pred, p10, p50b, p90b, p_r1)
        r1_day_files_zs.append(p_r1)

    merge_daily_route1(r1_day_files_zs, MERGED_R1_ZS)
    merge_daily_route2(r2_day_files_zs, MERGED_R2_ZS)
    print("✅ ZS done")

    del pipe_zs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =========================
# 10) PART B: FT Chronos-T5-small fine-tuned via AutoGluon
# =========================
if os.path.exists(MERGED_R1_FT) and os.path.exists(MERGED_R2_FT):
    print("\n✅ FT merged outputs already exist. Skipping FT.")
else:
    print("\n" + "=" * 80)
    print("PART B) FT asymmetric hourly: Chronos-T5-small fine-tuned via AutoGluon")

    from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

    if os.path.exists(LOCAL_TRAIN_DIR):
        shutil.rmtree(LOCAL_TRAIN_DIR)

    TARGET_COL = "power_w"

    # strict deployment-visible covariates only
    KNOWN_COVARIATES = [
        "temp_c",
        "hour",
        "dayofweek",
    ]

    def build_known_covariates_from_future_df(predictor, hist_ts, source_future_df, known_covariate_cols):
        future_index = predictor.make_future_data_frame(hist_ts).reset_index()

        future_cov = future_index.merge(
            source_future_df[["timestamp"] + known_covariate_cols],
            on="timestamp",
            how="left"
        )

        future_cov["item_id"] = future_cov["item_id"].astype(str)

        missing_counts = future_cov[known_covariate_cols].isna().sum()
        if missing_counts.sum() > 0:
            raise ValueError(
                "Missing future known covariates after alignment:\n"
                f"{missing_counts[missing_counts > 0]}"
            )

        known_cov_ts = TimeSeriesDataFrame.from_data_frame(
            future_cov[["item_id", "timestamp"] + known_covariate_cols],
            id_column="item_id",
            timestamp_column="timestamp"
        )

        return known_cov_ts

    # ---- train TS on hourly V1 with deployment-visible covariates only
    train_ag = train_df.copy().rename(columns={TARGET_COL: "target"})
    train_ag["item_id"] = "series_1"

    train_ts = TimeSeriesDataFrame.from_data_frame(
        train_ag[["item_id", "timestamp", "target"] + KNOWN_COVARIATES],
        id_column="item_id",
        timestamp_column="timestamp"
    )

    predictor = TimeSeriesPredictor(
        prediction_length=PRED_LEN,
        freq="1h",
        path=LOCAL_TRAIN_DIR,
        target="target",
        eval_metric="MASE",
        known_covariates_names=KNOWN_COVARIATES,
        quantile_levels=[Q_LO, Q_MID, Q_HI],
        verbosity=0,
    )

    t_fit = time.time()
    with suppress_output():
        predictor = predictor.fit(
            train_data=train_ts,
            hyperparameters={"Chronos": {"model_path": MODEL_FT_BASE, "fine_tune": True, "fine_tune_steps": FT_STEPS, "fine_tune_lr": FT_LR}},
            enable_ensemble=False,
        )

    best_model = predictor.model_best
    print(f"✅ FT trained in {time.time()-t_fit:.1f}s | best={best_model}")

    # ---- FT Route1 residual buckets from degraded hourly validation only
    rows_ft = []
    val_df = val_df.sort_values("timestamp").reset_index(drop=True)

    candidate_indices_raw = list(range(PRED_LEN, len(val_df) - PRED_LEN + 1, FT_CALIB_STRIDE))
    candidate_indices = maybe_limit_indices(candidate_indices_raw, FT_CALIB_MAX_WINDOWS)

    print(f"FT calibration windows used: {len(candidate_indices)}")

    for n, i in enumerate(candidate_indices, start=1):
        if n % 100 == 0:
            print(f"  FT calibration progress: {n}/{len(candidate_indices)}")

        hist = pd.concat(
            [train_df, val_df.iloc[:i]],
            ignore_index=True
        ).sort_values("timestamp").reset_index(drop=True)

        hist_ag = hist.copy().rename(columns={TARGET_COL: "target"})
        hist_ag["item_id"] = "series_1"

        hist_ts = TimeSeriesDataFrame.from_data_frame(
            hist_ag[["item_id", "timestamp", "target"] + KNOWN_COVARIATES],
            id_column="item_id",
            timestamp_column="timestamp"
        )

        source_future_df = val_df.iloc[i:i + PRED_LEN].copy().reset_index(drop=True)
        source_future_df["item_id"] = "series_1"

        try:
            known_cov_ts = build_known_covariates_from_future_df(
                predictor=predictor,
                hist_ts=hist_ts,
                source_future_df=source_future_df,
                known_covariate_cols=KNOWN_COVARIATES
            )
        except ValueError:
            continue

        with suppress_output():
            fc = predictor.predict(
                hist_ts,
                known_covariates=known_cov_ts,
                model=best_model
            )

        fc_pd = fc.reset_index().sort_values("timestamp").reset_index(drop=True)

        ts = pd.to_datetime(fc_pd.loc[0, "timestamp"])
        pred_q50 = float(fc_pd.loc[0, str(Q_MID)])
        y_t = float(val_df.loc[i, "power_w"])

        rows_ft.append({
            "hour": int(ts.hour),
            "dayofweek": int(ts.dayofweek),
            "residual": float(y_t - pred_q50)
        })

    df_resid_ft = pd.DataFrame(rows_ft)
    print("✅ FT residual points:", len(df_resid_ft))

    res_q_ft = (
        df_resid_ft.groupby(["hour", "dayofweek"])["residual"]
                   .quantile([Q_LO, Q_MID, Q_HI])
                   .unstack()
                   .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
                   .reset_index()
    )
    res_q_ft.to_csv(os.path.join(OUT_DIR, "chronos_trained_route1_residual_quantiles_from_v2val_asymmetric_hourly.csv"), index=False)

    # ---- FT test
    r1_day_files_ft = []
    r2_day_files_ft = []

    hist_master = pd.concat([train_df, val_df, test_df_all], ignore_index=True)
    hist_master = hist_master.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print(f"FT test day {k}: {day_start.date()}")

        day_end = day_start + pd.Timedelta(days=1)

        test_df = test_df_all[
            (test_df_all["timestamp"] >= day_start) &
            (test_df_all["timestamp"] < day_end)
        ].copy().reset_index(drop=True)

        y_true = test_df["power_w"].to_numpy(dtype=float)

        cutoff = day_start - pd.Timedelta(hours=1)
        hist_slice = hist_master[hist_master["timestamp"] <= cutoff].copy().reset_index(drop=True)

        hist_ag = hist_slice.copy().rename(columns={TARGET_COL: "target"})
        hist_ag["item_id"] = "series_1"

        hist_ts = TimeSeriesDataFrame.from_data_frame(
            hist_ag[["item_id", "timestamp", "target"] + KNOWN_COVARIATES],
            id_column="item_id",
            timestamp_column="timestamp"
        )

        source_future_df = test_df.copy().reset_index(drop=True)
        source_future_df["item_id"] = "series_1"

        known_cov_ts = build_known_covariates_from_future_df(
            predictor=predictor,
            hist_ts=hist_ts,
            source_future_df=source_future_df,
            known_covariate_cols=KNOWN_COVARIATES
        )

        with suppress_output():
            fc = predictor.predict(
                hist_ts,
                known_covariates=known_cov_ts,
                model=best_model
            )

        fc_pd = fc.reset_index().sort_values("timestamp").head(PRED_LEN).reset_index(drop=True)

        q10 = fc_pd[str(Q_LO)].to_numpy(dtype=float)
        q50 = fc_pd[str(Q_MID)].to_numpy(dtype=float)
        q90 = fc_pd[str(Q_HI)].to_numpy(dtype=float)
        q10, q50, q90 = enforce_non_crossing(q10, q50, q90)

        p_r2 = os.path.join(OUT_DIR, f"trained_route2_asymmetric_hourly_day{k}.csv")
        save_route2_day(test_df, y_true, q10, q50, q90, p_r2)
        r2_day_files_ft.append(p_r2)

        point_pred = q50.copy()
        p10, p50b, p90b = build_route1_buckets_hourly(test_df["timestamp"].values, point_pred, res_q_ft)

        p_r1 = os.path.join(OUT_DIR, f"trained_route1_asymmetric_hourly_day{k}.csv")
        save_route1_day(test_df, y_true, point_pred, p10, p50b, p90b, p_r1)
        r1_day_files_ft.append(p_r1)

    merge_daily_route1(r1_day_files_ft, MERGED_R1_FT)
    merge_daily_route2(r2_day_files_ft, MERGED_R2_FT)
    print("✅ FT done")

print("\nAll done.")
print("ZS Route1:", MERGED_R1_ZS)
print("ZS Route2:", MERGED_R2_ZS)
print("FT Route1:", MERGED_R1_FT)
print("FT Route2:", MERGED_R2_FT)